# DSN AI Bootcamp 2026 — Machine Learning Qualification Hackathon
## DSN Mart Product–Store Sales Prediction

**Objective:** Predict `total_sales` for each product–store observation in `test.csv`.

**Competition metric:** RMSE (Root Mean Squared Error). Lower RMSE indicates smaller prediction error.

### Execution plan
1. Load and audit the competition data.
2. Perform focused EDA that informs modelling.
3. Establish internal cross-validation using only `train.csv`.
4. Compare a mean baseline, Ridge regression, and two CatBoost configurations.
5. Select the model with the lowest mean cross-validation RMSE.
6. Refit the selected model on all training data.
7. Generate and validate the Kaggle submission file.
8. Save reproducibility outputs under `outputs/`.

This notebook is deliberately focused on completing a valid, defensible competition submission efficiently. Further experiments should only be added if time remains and they improve validation RMSE under the same validation protocol.

## 1. Setup and configuration

**Expected folder layout**

The notebook automatically checks both of these layouts:

```text
DSN AI Bootcamp/
├── DSN_2026_ML_Qualification_Hackathon.ipynb
├── train.csv
├── test.csv
└── sample_submission.csv
```

or

```text
DSN AI Bootcamp/
├── DSN_2026_ML_Qualification_Hackathon.ipynb
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

Keep the three CSV files unchanged.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

try:
    from catboost import CatBoostRegressor
except ImportError:
    raise ImportError(
        "CatBoost is not installed. In Anaconda Prompt run: pip install catboost"
    )

RANDOM_STATE = 42
N_FOLDS = 5
TARGET = "total_sales"
ID_COL = "id"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Find the CSV files either beside the notebook or in a data/ folder.
candidate_dirs = [Path("."), Path("data")]
DATA_DIR = next(
    (d for d in candidate_dirs
     if (d / "train.csv").exists()
     and (d / "test.csv").exists()
     and (d / "sample_submission.csv").exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find train.csv, test.csv and sample_submission.csv. "
        "Place them beside the notebook or inside a data/ folder."
    )

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


## 2. Load the competition files

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)


In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSubmission columns:")
print(sample_submission.columns.tolist())

display(train.head())
display(test.head())
display(sample_submission.head())


## 3. Basic data audit

The audit checks dimensions, data types, missingness, duplicates and target availability. The goal is to understand the modelling problem before choosing a model.

In [ ]:
audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "n_unique": train.nunique(dropna=False)
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

print("Duplicate train rows:", train.duplicated().sum())
print("Duplicate test rows:", test.duplicated().sum())
print("Target missing values:", train[TARGET].isna().sum())

display(audit)


## 4. Train/test structure

Because this is a product–store prediction problem, we check whether stores and products seen in the test set are represented in training data. We also check whether exact product–store pairs overlap.

`id` is treated as an identifier, not a predictor.

In [ ]:
train_products = set(train["product_code"].dropna().unique())
test_products = set(test["product_code"].dropna().unique())
train_stores = set(train["store_code"].dropna().unique())
test_stores = set(test["store_code"].dropna().unique())

train_pairs = set(zip(train["product_code"], train["store_code"]))
test_pairs = set(zip(test["product_code"], test["store_code"]))

print("Unique products in train:", len(train_products))
print("Unique products in test:", len(test_products))
print("Test products unseen in train:", len(test_products - train_products))
print("Stores in train:", len(train_stores))
print("Stores in test:", len(test_stores))
print("Test stores unseen in train:", len(test_stores - train_stores))
print("Exact product-store pairs shared by train/test:", len(train_pairs & test_pairs))
print("Exact product-store pairs in test:", len(test_pairs))


## 5. Focused EDA

Only analyses that can inform modelling are retained:
- target distribution
- store-level sales differences
- product-price relationship
- missingness

Large store-level differences suggest that store variables should be retained by the model. CatBoost is suitable because it can model nonlinear relationships and categorical variables directly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(train[TARGET], kde=True, ax=ax)
ax.set_title("Distribution of Total Sales")
ax.set_xlabel("Total sales")
plt.show()

store_summary = (
    train.groupby(["store_code", "store_format"], dropna=False)[TARGET]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)
display(store_summary)

category_summary = (
    train.groupby("product_category", dropna=False)[TARGET]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)
display(category_summary.head(15))

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=train, x="product_price", y=TARGET, alpha=0.35, ax=ax)
ax.set_title("Product Price vs Total Sales")
plt.show()

missing_summary = train.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(train) * 100).round(2)
display(missing_summary[missing_summary["missing_count"] > 0])


## 6. Validation strategy

Kaggle's test target is hidden, so model selection must use an internal validation procedure based only on `train.csv`.

We use **5-fold shuffled KFold cross-validation with `random_state=42`**. Every row is used for validation exactly once, while each model is trained only on the corresponding training folds.

The same folds are used for all model comparisons so that RMSE values are comparable.

In [ ]:
cv = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Validation: {N_FOLDS}-fold shuffled KFold, random_state={RANDOM_STATE}")


## 7. Mean baseline

In [ ]:
y = train[TARGET].copy()
baseline_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(train), start=1):
    y_tr = y.iloc[train_idx]
    y_va = y.iloc[valid_idx]
    prediction = np.repeat(y_tr.mean(), len(y_va))
    score = rmse(y_va, prediction)
    baseline_scores.append(score)

baseline_mean = np.mean(baseline_scores)
baseline_std = np.std(baseline_scores, ddof=0)

print(f"Mean baseline CV RMSE: {baseline_mean:.4f} ± {baseline_std:.4f}")


## 8. Ridge regression baseline

Ridge provides a conventional linear reference. Numeric variables are imputed and scaled; categorical variables are imputed and one-hot encoded.

This is a benchmark, not necessarily the final model.

In [ ]:
X = train.drop(columns=[TARGET, ID_COL]).copy()

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

ridge_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X), start=1):
    model = Pipeline([
        ("preprocessor", ridge_preprocessor),
        ("model", Ridge(alpha=10.0))
    ])
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[valid_idx])
    ridge_scores.append(rmse(y.iloc[valid_idx], pred))

ridge_mean = np.mean(ridge_scores)
ridge_std = np.std(ridge_scores, ddof=0)

print(f"Ridge CV RMSE: {ridge_mean:.4f} ± {ridge_std:.4f}")


## 9. CatBoost models

CatBoost is used because the dataset contains several categorical variables, including high-cardinality `product_code` and `store_code`. It can model categorical effects and nonlinear relationships without requiring one-hot encoding of every category.

Two closely related configurations are evaluated. The second is a modest tuning step rather than a broad hyperparameter search.

In [ ]:
X_cb = train.drop(columns=[TARGET, ID_COL]).copy()

cat_cols = X_cb.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    X_cb[col] = X_cb[col].fillna("Missing").astype(str)

catboost_configs = {
    "CatBoost_Base": {
        "iterations": 500,
        "depth": 7,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    "CatBoost_Tuned": {
        "iterations": 800,
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5
    }
}

catboost_results = {}
catboost_fold_scores = {}

for name, params in catboost_configs.items():
    scores = []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_cb), start=1):
        model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        )

        model.fit(
            X_cb.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_cols
        )

        pred = model.predict(X_cb.iloc[valid_idx])
        scores.append(rmse(y.iloc[valid_idx], pred))

    catboost_fold_scores[name] = scores
    catboost_results[name] = {
        "CV_RMSE_Mean": np.mean(scores),
        "CV_RMSE_Std": np.std(scores, ddof=0),
        "Runtime_seconds": time.time() - start
    }

catboost_results_df = pd.DataFrame(catboost_results).T.sort_values("CV_RMSE_Mean")
display(catboost_results_df)


## 10. Model comparison and selection

The final model is selected **only by the internal mean CV RMSE**. No leaderboard result is used to justify a model before submission.

This avoids choosing a model based on a subjective impression.

In [ ]:
comparison_rows = [
    ["Mean baseline", baseline_mean, baseline_std],
    ["Ridge", ridge_mean, ridge_std],
]

for name in catboost_results_df.index:
    comparison_rows.append([
        name,
        catboost_results[name]["CV_RMSE_Mean"],
        catboost_results[name]["CV_RMSE_Std"]
    ])

model_comparison = pd.DataFrame(
    comparison_rows,
    columns=["Model", "CV_RMSE_Mean", "CV_RMSE_Std"]
).sort_values("CV_RMSE_Mean")

display(model_comparison)

selected_model_name = model_comparison.iloc[0]["Model"]
print("Selected by CV RMSE:", selected_model_name)

# Save reproducibility record
model_comparison.to_csv(OUTPUT_DIR / "cv_model_comparison.csv", index=False)


## 11. Experiment log

This records what was tested, why it was tested, and how the decision was made.

In [ ]:
experiment_log = pd.DataFrame([
    {
        "Experiment": "E00",
        "Change": "Mean prediction baseline",
        "Reason": "Establish a simple reference point",
        "Validation": "5-fold KFold RMSE",
        "Result": baseline_mean,
        "Decision": "Reference only"
    },
    {
        "Experiment": "E01",
        "Change": "Ridge regression with one-hot categorical variables",
        "Reason": "Conventional linear benchmark",
        "Validation": "5-fold KFold RMSE",
        "Result": ridge_mean,
        "Decision": "Compare with nonlinear model"
    },
    {
        "Experiment": "E02",
        "Change": "CatBoost base configuration",
        "Reason": "Handle categorical variables and nonlinear effects",
        "Validation": "5-fold KFold RMSE",
        "Result": catboost_results["CatBoost_Base"]["CV_RMSE_Mean"],
        "Decision": "Retain as candidate"
    },
    {
        "Experiment": "E03",
        "Change": "CatBoost modest tuning",
        "Reason": "Test a deeper/shallower and learning-rate trade-off without broad search",
        "Validation": "5-fold KFold RMSE",
        "Result": catboost_results["CatBoost_Tuned"]["CV_RMSE_Mean"],
        "Decision": "Select if lowest CV RMSE"
    }
])

display(experiment_log)
experiment_log.to_csv(OUTPUT_DIR / "experiment_log.csv", index=False)


## 12. Final training and submission

The selected CatBoost configuration is retrained on **all available training rows** before predicting the hidden test set.

The notebook then validates the submission format:
- same number of rows as `test.csv`
- same IDs and order as `sample_submission.csv`
- no missing predictions
- no infinite predictions
- exactly the required columns: `id`, `total_sales`

The resulting file is saved as `outputs/submission.csv`.

In [ ]:
# Map the selected model name to its CatBoost configuration.
selected_params = catboost_configs.get(selected_model_name)

if selected_params is None:
    # The two CatBoost models are expected to outperform the simple baselines.
    # If a baseline wins unexpectedly, stop rather than silently submitting an incompatible model.
    raise RuntimeError(
        f"Selected model is {selected_model_name}. "
        "Review the CV results before final training."
    )

final_X = train.drop(columns=[TARGET, ID_COL]).copy()
final_test_X = test.drop(columns=[ID_COL]).copy()

for col in cat_cols:
    final_X[col] = final_X[col].fillna("Missing").astype(str)
    final_test_X[col] = final_test_X[col].fillna("Missing").astype(str)

final_model = CatBoostRegressor(
    **selected_params,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1
)

start = time.time()
final_model.fit(
    final_X,
    y,
    cat_features=cat_cols
)

test_predictions = final_model.predict(final_test_X)
training_seconds = time.time() - start

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_predictions
})

# Strict submission checks
assert len(submission) == len(test)
assert submission[ID_COL].equals(test[ID_COL])
assert submission.columns.tolist() == [ID_COL, TARGET]
assert submission[TARGET].notna().all()
assert np.isfinite(submission[TARGET]).all()

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Final model: {selected_model_name}")
print(f"Full-data training time: {training_seconds:.2f} seconds")
print(f"Submission saved to: {submission_path.resolve()}")
display(submission.head(10))


## 13. Final submission checks

Before uploading to Kaggle, inspect the generated file and confirm that the target predictions are numeric and the row count matches the test set.

In [ ]:
print("Submission shape:", submission.shape)
print("Expected rows:", len(test))
print("Columns:", submission.columns.tolist())
print("Missing predictions:", submission[TARGET].isna().sum())
print("Infinite predictions:", np.isinf(submission[TARGET]).sum())
print("Prediction summary:")
display(submission[TARGET].describe())

print("\nFile:")
print(submission_path.resolve())


## 14. Reproducibility summary

The following files are saved in `outputs/`:

- `submission.csv` — Kaggle-ready predictions.
- `cv_model_comparison.csv` — cross-validation comparison.
- `experiment_log.csv` — concise record of the modelling decisions.

For the final project record, also note the Kaggle submission date/time and the leaderboard RMSE returned after submission. The leaderboard value is an external evaluation and should be recorded separately from internal CV RMSE.

In [ ]:
print("Project run completed.")
print("\nKey internal result:")
display(model_comparison)

print("\nSaved outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


# End of notebook

### Submission action
Upload:

```text
outputs/submission.csv
```

to the DSN Kaggle competition submission page.

Then record:
1. submission timestamp
2. Kaggle public leaderboard RMSE
3. submission version/description

Do not replace the reproducible notebook result with a leaderboard-driven narrative. Keep internal validation and external Kaggle evaluation clearly separated.